# R-GCN + MILP: Tedarik Zincirinde Aday Hat Budama

Bu notebook, **Relational Graph Convolutional Network (R-GCN)** modelini gerçek bir yöneylem araştırması akışında kullanır.

Amaç GNN'nin MILP solver'ın yerine geçmesi değildir. Akış:

```text
tedarik zinciri instance'ı
        ↓
tam MILP
        ↓
optimal çözümde kullanılan hatlar = eğitim etiketi
        ↓
R-GCN edge scorer
        ↓
umut vermeyen aday hatları buda
        ↓
daha küçük MILP
        ↓
feasibility + objective gap + süre kontrolü
```

Ağ dört kademelidir:

```text
Supplier --supplies--> Plant --feeds--> Warehouse --ships_to--> Customer
```

R-GCN'de bu üç ilişki türü ayrı relation olarak modellenir. Bilginin iki yönde akabilmesi için ters ilişkiler de eklenir:

```text
rev_supplies, rev_feeds, rev_ships_to
```

> Bu örnek **candidate arc pruning / variable screening** fikrini öğretmek içindir. Gerçek bir üretim sisteminde pruning politikasının güvenli fallback mekanizması, distribution shift testi ve solver benchmark'ı zorunludur.


In [ ]:
import random
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from scipy.optimize import milp, LinearConstraint, Bounds
from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Sentetik çok kademeli tedarik zinciri instance'ı

Her instance'ta:

- 3 tedarikçi,
- 3 fabrika,
- 3 depo,
- 4 müşteri

bulunur.

Her ardışık katman arasında tüm aday hatlar başlangıçta mevcuttur. Maliyetler, kapasiteler ve talepler instance'a göre değişir.

Karar değişkenleri:

- $f_a \ge 0$: hat $a$ üzerindeki akış,
- $y_a\in\{0,1\}$: hatın kullanıma açılıp açılmadığı.

Amaç:

$$
\min \sum_a c_a f_a + \sum_a F_a y_a
$$

Bağlantı kısıtı:

$$
f_a \le U_a y_a.
$$

Buna ek olarak tedarikçi kapasiteleri, fabrika/depo akış dengeleri ve müşteri talepleri bulunur.


In [ ]:
N_SUPPLIERS = 3
N_PLANTS = 3
N_WAREHOUSES = 3
N_CUSTOMERS = 4

RELATION_NAMES = {
    0: "supplies",
    1: "feeds",
    2: "ships_to",
    3: "rev_supplies",
    4: "rev_feeds",
    5: "rev_ships_to",
}
NUM_RELATIONS = len(RELATION_NAMES)


def generate_instance(seed=0):
    rng = np.random.default_rng(seed)

    s0 = 0
    p0 = s0 + N_SUPPLIERS
    w0 = p0 + N_PLANTS
    c0 = w0 + N_WAREHOUSES

    suppliers = list(range(s0, p0))
    plants = list(range(p0, w0))
    warehouses = list(range(w0, c0))
    customers = list(range(c0, c0 + N_CUSTOMERS))

    node_types = np.array(
        [0] * N_SUPPLIERS
        + [1] * N_PLANTS
        + [2] * N_WAREHOUSES
        + [3] * N_CUSTOMERS
    )
    n_nodes = len(node_types)

    # Katman x koordinatı sabit, y koordinatı instance'a göre değişken.
    x_coord = np.array(
        [0] * N_SUPPLIERS
        + [1] * N_PLANTS
        + [2] * N_WAREHOUSES
        + [3] * N_CUSTOMERS,
        dtype=float,
    )
    y_coord = rng.uniform(0.0, 1.0, size=n_nodes)
    coords = np.column_stack([x_coord / 3.0, y_coord])

    demand = rng.integers(5, 13, size=N_CUSTOMERS).astype(float)
    total_demand = float(demand.sum())

    # Toplam kapasite talebin üzerinde tutulur.
    supplier_raw = rng.uniform(0.8, 1.2, size=N_SUPPLIERS)
    supply = total_demand * 1.45 * supplier_raw / supplier_raw.sum()

    plant_raw = rng.uniform(0.8, 1.2, size=N_PLANTS)
    plant_capacity = total_demand * 1.45 * plant_raw / plant_raw.sum()

    warehouse_raw = rng.uniform(0.8, 1.2, size=N_WAREHOUSES)
    warehouse_capacity = (
        total_demand * 1.45 * warehouse_raw / warehouse_raw.sum()
    )

    arcs = []

    def add_layer(src_nodes, dst_nodes, relation):
        for src in src_nodes:
            for dst in dst_nodes:
                distance = float(np.linalg.norm(coords[src] - coords[dst]))

                variable_cost = (
                    1.0
                    + 8.0 * distance
                    + rng.uniform(0.0, 2.0)
                )
                fixed_cost = (
                    1.0
                    + 5.0 * distance
                    + rng.uniform(0.0, 3.0)
                )
                arc_capacity = total_demand * rng.uniform(0.42, 0.75)

                arcs.append(
                    {
                        "src": src,
                        "dst": dst,
                        "relation": relation,
                        "variable_cost": variable_cost,
                        "fixed_cost": fixed_cost,
                        "capacity": arc_capacity,
                    }
                )

    add_layer(suppliers, plants, relation=0)
    add_layer(plants, warehouses, relation=1)
    add_layer(warehouses, customers, relation=2)

    return {
        "suppliers": suppliers,
        "plants": plants,
        "warehouses": warehouses,
        "customers": customers,
        "node_types": node_types,
        "coords": coords,
        "demand": demand,
        "total_demand": total_demand,
        "supply": supply,
        "plant_capacity": plant_capacity,
        "warehouse_capacity": warehouse_capacity,
        "arcs": arcs,
    }


example = generate_instance(SEED)
len(example["arcs"]), example["demand"], example["total_demand"]


## 2. Tam ve budanmış ağ için MILP solver

`scipy.optimize.milp`, HiGHS tabanlı bir MILP arayüzüdür. Burada ücretsiz ve kurulumu kolay bir baseline olması için kullanıyoruz.

Bir `keep_mask` verilirse yalnızca seçilen aday hatlar MILP'ye dahil edilir. Böylece GNN budamasının model boyutuna ve çözüme etkisini doğrudan ölçebiliriz.


In [ ]:
def solve_supply_chain(inst, keep_mask=None, time_limit=10.0):
    arcs = inst["arcs"]

    if keep_mask is None:
        keep_idx = list(range(len(arcs)))
    else:
        keep_idx = np.flatnonzero(
            np.asarray(keep_mask, dtype=bool)
        ).tolist()

    m = len(keep_idx)
    if m == 0:
        return None

    # Değişken sırası:
    # [flow_0, ..., flow_(m-1), y_0, ..., y_(m-1)]
    objective = np.zeros(2 * m, dtype=float)

    lower = np.zeros(2 * m, dtype=float)
    upper = np.empty(2 * m, dtype=float)
    integrality = np.zeros(2 * m, dtype=int)

    for local_k, global_k in enumerate(keep_idx):
        arc = arcs[global_k]

        objective[local_k] = arc["variable_cost"]
        objective[m + local_k] = arc["fixed_cost"]

        upper[local_k] = arc["capacity"]
        upper[m + local_k] = 1.0
        integrality[m + local_k] = 1

    rows = []
    lower_cons = []
    upper_cons = []

    def add_constraint(coefficients, lb, ub):
        rows.append(coefficients)
        lower_cons.append(lb)
        upper_cons.append(ub)

    # flow_a <= capacity_a * y_a
    for local_k, global_k in enumerate(keep_idx):
        arc = arcs[global_k]
        row = np.zeros(2 * m, dtype=float)

        row[local_k] = 1.0
        row[m + local_k] = -arc["capacity"]

        add_constraint(row, -np.inf, 0.0)

    def flow_row(predicate):
        row = np.zeros(2 * m, dtype=float)
        for local_k, global_k in enumerate(keep_idx):
            row[local_k] = predicate(arcs[global_k])
        return row

    # Supplier outbound <= supply
    for idx, supplier in enumerate(inst["suppliers"]):
        row = flow_row(
            lambda a, supplier=supplier:
            1.0 if a["src"] == supplier else 0.0
        )
        add_constraint(row, -np.inf, inst["supply"][idx])

    # Plant: inbound = outbound ve throughput <= capacity
    for idx, plant in enumerate(inst["plants"]):
        balance = flow_row(
            lambda a, plant=plant:
            (1.0 if a["dst"] == plant else 0.0)
            - (1.0 if a["src"] == plant else 0.0)
        )
        add_constraint(balance, 0.0, 0.0)

        inbound = flow_row(
            lambda a, plant=plant:
            1.0 if a["dst"] == plant else 0.0
        )
        add_constraint(
            inbound,
            -np.inf,
            inst["plant_capacity"][idx],
        )

    # Warehouse: inbound = outbound ve throughput <= capacity
    for idx, warehouse in enumerate(inst["warehouses"]):
        balance = flow_row(
            lambda a, warehouse=warehouse:
            (1.0 if a["dst"] == warehouse else 0.0)
            - (1.0 if a["src"] == warehouse else 0.0)
        )
        add_constraint(balance, 0.0, 0.0)

        inbound = flow_row(
            lambda a, warehouse=warehouse:
            1.0 if a["dst"] == warehouse else 0.0
        )
        add_constraint(
            inbound,
            -np.inf,
            inst["warehouse_capacity"][idx],
        )

    # Her müşteri talebi tam karşılanır.
    for idx, customer in enumerate(inst["customers"]):
        inbound = flow_row(
            lambda a, customer=customer:
            1.0 if a["dst"] == customer else 0.0
        )
        add_constraint(
            inbound,
            inst["demand"][idx],
            inst["demand"][idx],
        )

    A = np.vstack(rows)
    constraints = LinearConstraint(
        A,
        np.asarray(lower_cons, dtype=float),
        np.asarray(upper_cons, dtype=float),
    )

    start = time.perf_counter()

    result = milp(
        c=objective,
        integrality=integrality,
        bounds=Bounds(lower, upper),
        constraints=constraints,
        options={
            "time_limit": time_limit,
            "presolve": True,
        },
    )

    elapsed = time.perf_counter() - start

    if not result.success:
        return None

    flows = np.zeros(len(arcs), dtype=float)
    y = np.zeros(len(arcs), dtype=float)

    for local_k, global_k in enumerate(keep_idx):
        flows[global_k] = result.x[local_k]
        y[global_k] = result.x[m + local_k]

    return {
        "objective": float(result.fun),
        "flows": flows,
        "y": y,
        "keep_idx": keep_idx,
        "elapsed": elapsed,
        "mip_node_count": getattr(result, "mip_node_count", None),
        "mip_gap": getattr(result, "mip_gap", None),
    }


full_solution = solve_supply_chain(example)
full_solution["objective"], int(full_solution["y"].sum())


## 3. Eğitim etiketlerini MILP optimumundan üretme

Her eğitim instance'ında tam aday ağı çözüyoruz.

Bir hat optimal çözümde açılmışsa:

$$
y_a^*=1
$$

etiketi verilir.

Bu yaklaşım **supervised imitation / learning from optimization** örneğidir. Burada expert, tam MILP solver'dır.

> Etiket üretmenin maliyetli olması gerçek projelerde önemli bir sorundur. Daha büyük problemlerde solver log'ları, incumbents, relaxations, pseudo-labeling veya self-supervised hedefler kullanılabilir.


In [ ]:
def make_labeled_instances(
    start_seed,
    count,
    max_attempts_per_instance=20,
):
    instances = []
    seed = start_seed

    while len(instances) < count:
        found = False

        for _ in range(max_attempts_per_instance):
            inst = generate_instance(seed)
            solution = solve_supply_chain(inst)

            seed += 1

            if solution is not None:
                inst["full_solution"] = solution
                instances.append(inst)
                found = True
                break

        if not found:
            raise RuntimeError(
                "Uygulanabilir instance üretilemedi."
            )

    return instances


train_instances = make_labeled_instances(1_000, 50)
val_instances = make_labeled_instances(2_000, 10)
test_instances = make_labeled_instances(3_000, 10)

print(
    len(train_instances),
    len(val_instances),
    len(test_instances),
)


## 4. Tedarik zincirini R-GCN graph'ına dönüştürme

Node feature'ları:

- node-type one-hot: supplier / plant / warehouse / customer,
- normalize kapasite,
- normalize talep,
- katman koordinatı,
- yatay/konumsal koordinat.

Forward relation türleri:

- `supplies`,
- `feeds`,
- `ships_to`.

Message passing'in aşağı ve yukarı akabilmesi için ters relation'lar eklenir.

Edge classifier ayrıca hatın:

- değişken maliyetini,
- sabit maliyetini,
- kapasitesini,
- relation one-hot bilgisini

kullanır.

Bu nedenle model yalnızca topolojiyi değil, optimizasyon katsayılarını da görür.


In [ ]:
def instance_to_graph(inst):
    n_nodes = len(inst["node_types"])
    total_demand = max(inst["total_demand"], 1.0)

    node_type_onehot = np.eye(4, dtype=float)[inst["node_types"]]

    resource_capacity = np.zeros(n_nodes, dtype=float)
    demand_feature = np.zeros(n_nodes, dtype=float)

    for idx, node in enumerate(inst["suppliers"]):
        resource_capacity[node] = inst["supply"][idx]

    for idx, node in enumerate(inst["plants"]):
        resource_capacity[node] = inst["plant_capacity"][idx]

    for idx, node in enumerate(inst["warehouses"]):
        resource_capacity[node] = inst["warehouse_capacity"][idx]

    for idx, node in enumerate(inst["customers"]):
        demand_feature[node] = inst["demand"][idx]

    node_x = np.column_stack(
        [
            node_type_onehot,
            resource_capacity / total_demand,
            demand_feature / total_demand,
            inst["coords"][:, 0],
            inst["coords"][:, 1],
        ]
    )

    forward_src = []
    forward_dst = []
    forward_rel = []
    edge_features = []

    variable_costs = np.array(
        [a["variable_cost"] for a in inst["arcs"]],
        dtype=float,
    )
    fixed_costs = np.array(
        [a["fixed_cost"] for a in inst["arcs"]],
        dtype=float,
    )

    max_variable_cost = max(variable_costs.max(), 1.0)
    max_fixed_cost = max(fixed_costs.max(), 1.0)

    for arc in inst["arcs"]:
        forward_src.append(arc["src"])
        forward_dst.append(arc["dst"])
        forward_rel.append(arc["relation"])

        relation_onehot = np.eye(3)[arc["relation"]]

        edge_features.append(
            np.concatenate(
                [
                    np.array(
                        [
                            arc["variable_cost"] / max_variable_cost,
                            arc["fixed_cost"] / max_fixed_cost,
                            arc["capacity"] / total_demand,
                        ]
                    ),
                    relation_onehot,
                ]
            )
        )

    forward_src = np.asarray(forward_src, dtype=int)
    forward_dst = np.asarray(forward_dst, dtype=int)
    forward_rel = np.asarray(forward_rel, dtype=int)

    # Forward ve reverse message-passing edge'leri.
    mp_src = np.concatenate([forward_src, forward_dst])
    mp_dst = np.concatenate([forward_dst, forward_src])
    mp_rel = np.concatenate(
        [
            forward_rel,
            forward_rel + 3,
        ]
    )

    data = Data(
        x=torch.tensor(node_x, dtype=torch.float32),
        edge_index=torch.tensor(
            np.vstack([mp_src, mp_dst]),
            dtype=torch.long,
        ),
        edge_type=torch.tensor(mp_rel, dtype=torch.long),
    )

    data.candidate_edge_index = torch.tensor(
        np.vstack([forward_src, forward_dst]),
        dtype=torch.long,
    )
    data.candidate_edge_attr = torch.tensor(
        np.asarray(edge_features),
        dtype=torch.float32,
    )
    data.edge_y = torch.tensor(
        inst["full_solution"]["y"],
        dtype=torch.float32,
    )

    return data


train_graphs = [
    instance_to_graph(inst)
    for inst in train_instances
]
val_graphs = [
    instance_to_graph(inst)
    for inst in val_instances
]
test_graphs = [
    instance_to_graph(inst)
    for inst in test_instances
]

train_graphs[0]


## 5. R-GCN edge scorer

R-GCN relation-specific dönüşüm kullanır:

$$
h_i^{(l+1)}
=
\sigma
\left(
W_0 h_i^{(l)}
+
\sum_{r\in\mathcal R}
\sum_{j\in N_i^r}
\frac{1}{c_{i,r}}
W_r h_j^{(l)}
\right).
$$

Burada 6 relation vardır: 3 forward + 3 reverse.

İki R-GCN katmanından sonra her aday forward arc için:

$$
[h_{src}, h_{dst}, e_a]
$$

birleştirilir ve MLP ile `arc kullanılmalı mı?` logiti üretilir.


In [ ]:
class SupplyChainRGCN(nn.Module):
    def __init__(
        self,
        node_dim,
        edge_dim,
        hidden_dim=64,
        num_relations=NUM_RELATIONS,
    ):
        super().__init__()

        self.input_proj = nn.Linear(
            node_dim,
            hidden_dim,
        )

        self.conv1 = RGCNConv(
            hidden_dim,
            hidden_dim,
            num_relations=num_relations,
            num_bases=3,
        )
        self.conv2 = RGCNConv(
            hidden_dim,
            hidden_dim,
            num_relations=num_relations,
            num_bases=3,
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.edge_head = nn.Sequential(
            nn.Linear(
                2 * hidden_dim + edge_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x = self.input_proj(data.x)

        h1 = self.conv1(
            x,
            data.edge_index,
            data.edge_type,
        )
        x = self.norm1(
            x + F.relu(h1)
        )

        h2 = self.conv2(
            x,
            data.edge_index,
            data.edge_type,
        )
        x = self.norm2(
            x + F.relu(h2)
        )

        src, dst = data.candidate_edge_index

        edge_repr = torch.cat(
            [
                x[src],
                x[dst],
                data.candidate_edge_attr,
            ],
            dim=-1,
        )

        return self.edge_head(edge_repr).squeeze(-1)


node_dim = train_graphs[0].x.shape[1]
edge_dim = train_graphs[0].candidate_edge_attr.shape[1]

model = SupplyChainRGCN(
    node_dim=node_dim,
    edge_dim=edge_dim,
).to(device)

model


## 6. Eğitim

Açılan hat sayısı tüm aday hat sayısından az olduğu için pozitif sınıf nispeten seyrektir. Bu nedenle `BCEWithLogitsLoss` içinde `pos_weight` kullanıyoruz.

Model seçiminde yalnız training loss'a değil validation loss'a bakıyoruz.


In [ ]:
def move_graph(data, device):
    return data.to(device)


all_train_labels = torch.cat(
    [g.edge_y for g in train_graphs]
)
n_pos = float(all_train_labels.sum())
n_neg = float(
    all_train_labels.numel()
    - all_train_labels.sum()
)

pos_weight = torch.tensor(
    n_neg / max(n_pos, 1.0),
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-3,
    weight_decay=1e-4,
)


def dataset_loss(graphs):
    model.eval()
    losses = []

    with torch.no_grad():
        for graph in graphs:
            graph = move_graph(graph, device)

            logits = model(graph)
            loss = criterion(
                logits,
                graph.edge_y,
            )

            losses.append(float(loss.item()))

    return float(np.mean(losses))


best_state = None
best_val = float("inf")

for epoch in range(1, 61):
    model.train()
    order = np.random.permutation(
        len(train_graphs)
    )
    train_losses = []

    for idx in order:
        graph = move_graph(
            train_graphs[idx],
            device,
        )

        optimizer.zero_grad()

        logits = model(graph)
        loss = criterion(
            logits,
            graph.edge_y,
        )

        loss.backward()
        optimizer.step()

        train_losses.append(
            float(loss.item())
        )

    if epoch % 5 == 0 or epoch == 1:
        val_loss = dataset_loss(
            val_graphs
        )

        if val_loss < best_val:
            best_val = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"epoch={epoch:02d} "
            f"train={np.mean(train_losses):.4f} "
            f"val={val_loss:.4f}"
        )


if best_state is not None:
    model.load_state_dict(best_state)

model.eval()


## 7. Edge-prediction metriği

Buradaki amaç bir classification yarışması kazanmak değildir. Yine de modelin expert MILP tarafından seçilen hatları ne kadar yakaladığını görmek gerekir.

Asıl OR metriği bir sonraki bölümde:

- prune sonrası feasibility,
- retained arc ratio,
- objective gap,
- solve time

olacaktır.


In [ ]:
def edge_classification_metrics(graphs, threshold=0.5):
    tp = fp = fn = tn = 0

    model.eval()

    with torch.no_grad():
        for graph in graphs:
            graph = move_graph(graph, device)

            prob = torch.sigmoid(
                model(graph)
            )
            pred = prob >= threshold
            target = graph.edge_y >= 0.5

            tp += int(
                (pred & target).sum().item()
            )
            fp += int(
                (pred & ~target).sum().item()
            )
            fn += int(
                (~pred & target).sum().item()
            )
            tn += int(
                (~pred & ~target).sum().item()
            )

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-12)
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


edge_classification_metrics(test_graphs)


## 8. GNN-guided candidate arc pruning

Naif biçimde `score < 0.5 olan her hattı sil` demek tehlikelidir. Bir müşteri erişilemez kalabilir veya ara katmanda akış yolu kopabilir.

Bu nedenle iki güvenlik mekanizması kullanıyoruz:

1. **Topolojik safeguard:** her kritik node için en az birkaç yüksek skorlu incoming/outgoing hat korunur.
2. **Adaptive fallback:** budanmış MILP infeasible ise silinen hatlar skora göre geri eklenir ve tekrar çözülür.

Bu fallback özellikle önemlidir:

> GNN pruning önerir; feasibility kararını solver verir.


In [ ]:
def initial_pruning_mask(
    scores,
    inst,
    keep_ratio=0.50,
    minimum_incident=2,
):
    scores = np.asarray(scores, dtype=float)
    n_arcs = len(inst["arcs"])

    keep_count = max(
        1,
        int(np.ceil(keep_ratio * n_arcs)),
    )

    ranking = np.argsort(-scores)

    keep = np.zeros(
        n_arcs,
        dtype=bool,
    )
    keep[ranking[:keep_count]] = True

    def force_best(indices, count):
        if not indices:
            return

        indices = np.asarray(indices, dtype=int)
        local_order = indices[
            np.argsort(-scores[indices])
        ]

        keep[
            local_order[
                : min(count, len(local_order))
            ]
        ] = True

    arcs = inst["arcs"]

    # Supplier: outbound
    for node in inst["suppliers"]:
        force_best(
            [
                k for k, a in enumerate(arcs)
                if a["src"] == node
            ],
            minimum_incident,
        )

    # Plant: inbound + outbound
    for node in inst["plants"]:
        force_best(
            [
                k for k, a in enumerate(arcs)
                if a["dst"] == node
            ],
            minimum_incident,
        )
        force_best(
            [
                k for k, a in enumerate(arcs)
                if a["src"] == node
            ],
            minimum_incident,
        )

    # Warehouse: inbound + outbound
    for node in inst["warehouses"]:
        force_best(
            [
                k for k, a in enumerate(arcs)
                if a["dst"] == node
            ],
            minimum_incident,
        )
        force_best(
            [
                k for k, a in enumerate(arcs)
                if a["src"] == node
            ],
            minimum_incident,
        )

    # Customer: inbound
    for node in inst["customers"]:
        force_best(
            [
                k for k, a in enumerate(arcs)
                if a["dst"] == node
            ],
            minimum_incident,
        )

    return keep


def adaptive_pruned_solve(
    inst,
    scores,
    keep_ratio=0.50,
    add_batch=3,
):
    scores = np.asarray(scores, dtype=float)

    keep = initial_pruning_mask(
        scores,
        inst,
        keep_ratio=keep_ratio,
    )

    omitted_order = [
        idx
        for idx in np.argsort(-scores)
        if not keep[idx]
    ]

    solution = solve_supply_chain(
        inst,
        keep_mask=keep,
    )

    cursor = 0

    while (
        solution is None
        and cursor < len(omitted_order)
    ):
        for _ in range(add_batch):
            if cursor >= len(omitted_order):
                break

            keep[omitted_order[cursor]] = True
            cursor += 1

        solution = solve_supply_chain(
            inst,
            keep_mask=keep,
        )

    return solution, keep


def model_scores(graph):
    model.eval()

    with torch.no_grad():
        graph = move_graph(graph, device)

        return (
            torch.sigmoid(model(graph))
            .detach()
            .cpu()
            .numpy()
        )


## 9. Güçlü olmayan ama gerekli baseline: sadece maliyetle budama

GNN'yi random veya çok zayıf baseline'a karşı kıyaslamak yeterli değildir.

Burada basit bir mühendislik heuristic'i kuruyoruz:

- düşük değişken maliyet,
- düşük sabit maliyet

daha yüksek skor alsın.

Eğer R-GCN bu basit baseline'dan sistematik biçimde daha iyi değilse modelin ek karmaşıklığı sorgulanmalıdır.


In [ ]:
def cost_heuristic_scores(inst):
    variable_cost = np.array(
        [
            a["variable_cost"]
            for a in inst["arcs"]
        ],
        dtype=float,
    )
    fixed_cost = np.array(
        [
            a["fixed_cost"]
            for a in inst["arcs"]
        ],
        dtype=float,
    )

    variable_cost = (
        variable_cost
        / max(variable_cost.max(), 1.0)
    )
    fixed_cost = (
        fixed_cost
        / max(fixed_cost.max(), 1.0)
    )

    # Düşük toplam normalize maliyet -> yüksek skor
    return -(
        0.7 * variable_cost
        + 0.3 * fixed_cost
    )


## 10. Test: tam MILP vs R-GCN pruning vs maliyet heuristic'i

Her test instance'ı için tam MILP yeniden çözülür. Sonra aynı `keep_ratio` ile:

- R-GCN scoring,
- cost-only scoring

uygulanır.

Ölçülen temel değerler:

$$
\text{objective gap} =
100
\frac{z_{\text{pruned}}-z_{\text{full}}}
{|z_{\text{full}}|}
$$

ve

$$
\text{arc retention}
=
\frac{\text{kalan aday hat}}
{\text{tam aday hat}}.
$$


In [ ]:
def evaluate_pruning(
    instances,
    graphs,
    keep_ratio=0.50,
):
    records = []

    for idx, (inst, graph) in enumerate(
        zip(instances, graphs)
    ):
        full = solve_supply_chain(inst)

        rgcn_score = model_scores(graph)
        cost_score = cost_heuristic_scores(inst)

        rgcn_sol, rgcn_keep = adaptive_pruned_solve(
            inst,
            rgcn_score,
            keep_ratio=keep_ratio,
        )

        cost_sol, cost_keep = adaptive_pruned_solve(
            inst,
            cost_score,
            keep_ratio=keep_ratio,
        )

        if full is None:
            continue

        def summarize(method, solution, keep):
            if solution is None:
                return {
                    "method": method,
                    "feasible": False,
                    "gap_pct": np.nan,
                    "retained_pct": 100.0 * keep.mean(),
                    "time": np.nan,
                }

            gap = (
                100.0
                * (
                    solution["objective"]
                    - full["objective"]
                )
                / max(abs(full["objective"]), 1e-9)
            )

            return {
                "method": method,
                "feasible": True,
                "gap_pct": gap,
                "retained_pct": 100.0 * keep.mean(),
                "time": solution["elapsed"],
            }

        full_record = {
            "instance": idx,
            "method": "full_milp",
            "feasible": True,
            "gap_pct": 0.0,
            "retained_pct": 100.0,
            "time": full["elapsed"],
        }

        rgcn_record = {
            "instance": idx,
            **summarize(
                "rgcn_pruning",
                rgcn_sol,
                rgcn_keep,
            ),
        }

        cost_record = {
            "instance": idx,
            **summarize(
                "cost_pruning",
                cost_sol,
                cost_keep,
            ),
        }

        records.extend(
            [
                full_record,
                rgcn_record,
                cost_record,
            ]
        )

    return records


records = evaluate_pruning(
    test_instances,
    test_graphs,
    keep_ratio=0.50,
)

for row in records[:9]:
    print(row)


In [ ]:
def aggregate_records(records):
    methods = sorted(
        {r["method"] for r in records}
    )

    for method in methods:
        rows = [
            r for r in records
            if r["method"] == method
        ]

        feasible_rate = np.mean(
            [r["feasible"] for r in rows]
        )

        gaps = np.array(
            [
                r["gap_pct"]
                for r in rows
                if np.isfinite(r["gap_pct"])
            ],
            dtype=float,
        )

        retained = np.array(
            [r["retained_pct"] for r in rows],
            dtype=float,
        )

        times = np.array(
            [
                r["time"]
                for r in rows
                if np.isfinite(r["time"])
            ],
            dtype=float,
        )

        print(
            f"{method:14s} | "
            f"feasible={feasible_rate:5.1%} | "
            f"gap={np.mean(gaps):7.3f}% | "
            f"arcs={np.mean(retained):6.1f}% | "
            f"time={np.mean(times):.5f}s"
        )


aggregate_records(records)


## 11. Sonuçları nasıl yorumlamalıyız?

Bu notebook'u çalıştırdığınızda R-GCN'nin her seferinde mükemmel pruning yapması beklenmemelidir. Asıl bilimsel sorular şunlardır:

1. Aynı objective gap düzeyinde kaç aday değişken/hat kaldırılıyor?
2. Feasibility fallback ne kadar sık devreye giriyor?
3. GNN inference süresi dahil edildiğinde toplam süre gerçekten düşüyor mu?
4. GNN, basit cost heuristic'ini geçiyor mu?
5. Eğitimden daha büyük veya farklı dağılımdaki ağlarda performans korunuyor mu?

Gerçek bir çalışma için benchmark tablosu şu şekilde olmalıdır:

```text
method
objective
optimality gap
MILP variables
MILP constraints
solver nodes
solver time
GNN inference time
total time
feasibility rate
```

### Neden bu örnek “matematiksel fantezi” değil?

Buradaki bileşenlerin her biri standart ve gerçek araçlardır:

- **R-GCN:** multi-relational graph message passing için yerleşik bir GNN.
- **MILP:** tedarik zinciri ağ tasarımı ve fixed-charge network flow için klasik OR yaklaşımı.
- **Candidate pruning / variable screening:** solver'a girmeden önce karar uzayını küçültmek için kullanılan learning-augmented optimization fikri.
- **Fallback solve:** ML kararının feasibility/correctness mekanizmasının yerine geçmemesini sağlar.

Ancak bu notebook'taki sentetik sonuçlar endüstriyel başarı kanıtı değildir. Gerçek veri, güçlü solver baseline'ı ve out-of-distribution test gereklidir.

## İleri geliştirme

Bu notebook şu yönlerde büyütülebilir:

- Pyomo + SCIP/Gurobi ile daha büyük network design,
- tesis açma binary değişkenleri,
- çok ürünlü akış,
- stok ve dönem boyutu ekleyerek multi-period model,
- disruption senaryoları,
- Graph Transformer veya RGAT ile karşılaştırma,
- learned Benders/decomposition guidance,
- uncertainty-aware pruning.
